# IF + Math Task Arithmetic Merge (FP32)

This notebook builds a merged checkpoint with classic task arithmetic:

\[\theta_{merge} = \theta_{base} + \lambda_{if} \cdot \Delta_{if} + \lambda_{math} \cdot \Delta_{math}\]

Task vector definition is intentionally kept identical to `08_jwcm_v2_soft_attribution_merge_fisher_rollout_reuse.ipynb`:

\[\Delta_{task} = \theta_{task} - \theta_{base}\]

All load/merge/save steps are enforced in FP32.

In [ ]:
from __future__ import annotations

import gc
import json
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List

import numpy as np
import torch


@dataclass(frozen=True)
class TaskSpec:
    """Task checkpoint specification used by task arithmetic merge.

    Args:
        name: Task identifier used in lambda mapping and artifact names.
        model_path: Local checkpoint path for the task model.
    """

    name: str
    model_path: Path


@dataclass(frozen=True)
class RuntimeConfig:
    """Runtime configuration for FP32 task arithmetic merge.

    Args:
        base_model_id: Base model id/path used as merge anchor.
        output_root: Directory where merged checkpoints will be saved.
        seed: Global random seed for reproducibility.
        model_dtype: Torch dtype used for load/merge/save. Must be float32.
        device: Runtime device string for merge execution.
        lambda_if: Scalar weight for IF task vector.
        lambda_math: Scalar weight for Math task vector.
    """

    base_model_id: str
    output_root: Path
    seed: int
    model_dtype: torch.dtype
    device: str
    lambda_if: float
    lambda_math: float


BASE_MODEL_ID = "Qwen/Qwen3-1.7B"
IF_MODEL_PATH = Path(
    "/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-ifrl_ifeval/global_step_50/actor/huggingface"
)
MATH_MODEL_PATH = Path(
    "/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-math/stage2/global_step_40/actor/huggingface"
)

TASK_SPECS: List[TaskSpec] = [
    TaskSpec(name="if", model_path=IF_MODEL_PATH),
    TaskSpec(name="math", model_path=MATH_MODEL_PATH),
]

RUNTIME = RuntimeConfig(
    base_model_id=BASE_MODEL_ID,
    output_root=Path(
        "/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-task-arithmetic-fp32"
    ),
    seed=42,
    model_dtype=torch.float32,
    device="cpu",
    lambda_if=1.0,
    lambda_math=1.0,
)

# Fail fast on required local task checkpoints so runtime errors are explicit.
for required_path in [task_spec.model_path for task_spec in TASK_SPECS]:
    if not required_path.exists():
        raise FileNotFoundError(f"Required task checkpoint does not exist: {required_path}")

# The user requested FP32 end-to-end (load, arithmetic, save), so we enforce it once here.
if RUNTIME.model_dtype != torch.float32:
    raise ValueError(
        "This notebook requires FP32 end-to-end for model load/merge/save. "
        f"Configured dtype={RUNTIME.model_dtype}"
    )

RUNTIME.output_root.mkdir(parents=True, exist_ok=True)
(RUNTIME.output_root / "metadata").mkdir(parents=True, exist_ok=True)

print(f"Base model id: {RUNTIME.base_model_id}")
print(f"Task checkpoints: {[str(task.model_path) for task in TASK_SPECS]}")
print(f"Task lambdas: if={RUNTIME.lambda_if}, math={RUNTIME.lambda_math}")
print(f"Runtime dtype (must be FP32): {RUNTIME.model_dtype}")
print(f"Output root: {RUNTIME.output_root}")

In [ ]:
from datetime import datetime
from typing import Any, Mapping

from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer


def set_seed(seed: int) -> None:
    """Set random seeds for reproducibility.

    Args:
        seed: Integer random seed.

    Returns:
        None. Global RNG states are updated in-place.
    """

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def now_iso() -> str:
    """Return current UTC timestamp for metadata.

    Returns:
        ISO-8601 timestamp string in UTC.
    """

    return datetime.utcnow().isoformat(timespec="seconds") + "Z"


def save_json(payload: Mapping[str, Any], output_path: Path) -> None:
    """Save JSON payload to disk with pretty formatting.

    Args:
        payload: JSON-serializable mapping object.
        output_path: Destination JSON path.

    Returns:
        None. File is written to disk.
    """

    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", encoding="utf-8") as file:
        json.dump(payload, file, indent=2, ensure_ascii=False)


def format_float_token(value: float) -> str:
    """Convert float to filesystem-safe token for artifact names.

    Args:
        value: Float value to encode.

    Returns:
        String token where `.` is replaced by `p` and `-` by `m`.
        Example: `1.25 -> 1p25`, `-0.5 -> m0p5`.
    """

    formatted = f"{float(value):.6f}".rstrip("0").rstrip(".")
    if formatted in {"", "-0"}:
        formatted = "0"
    return formatted.replace("-", "m").replace(".", "p")


def load_tokenizer_with_mistral_regex_fix(model_name_or_path: str) -> AutoTokenizer:
    """Load tokenizer with optional `fix_mistral_regex=True` fallback handling.

    Args:
        model_name_or_path: Hugging Face model id or local path.

    Returns:
        Loaded tokenizer with trust-remote-code enabled.
    """

    try:
        return AutoTokenizer.from_pretrained(
            model_name_or_path,
            trust_remote_code=True,
            fix_mistral_regex=True,
        )
    except TypeError:
        return AutoTokenizer.from_pretrained(
            model_name_or_path,
            trust_remote_code=True,
        )


def load_causal_lm_fp32(
    model_name_or_path: str | Path,
    device: str,
) -> tuple[AutoModelForCausalLM, AutoTokenizer]:
    """Load CausalLM + tokenizer in strict FP32.

    Args:
        model_name_or_path: Hugging Face model id or local checkpoint path.
        device: Runtime device string.

    Returns:
        Tuple of `(model, tokenizer)` loaded in eval mode.
    """

    resolved_path = str(model_name_or_path)

    # `torch_dtype=torch.float32` makes weight loading explicit.
    model = AutoModelForCausalLM.from_pretrained(
        resolved_path,
        torch_dtype=torch.float32,
        device_map=None,
        low_cpu_mem_usage=True,
        trust_remote_code=True,
    )

    # Explicit cast to FP32 again so arithmetic phase always starts from FP32 tensors.
    model.to(device=device, dtype=torch.float32)
    model.eval()

    tokenizer = load_tokenizer_with_mistral_regex_fix(resolved_path)
    if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
        tokenizer.pad_token = tokenizer.eos_token

    return model, tokenizer


def assert_model_float32(model: AutoModelForCausalLM, model_label: str) -> None:
    """Assert that every floating parameter in a model is FP32.

    Args:
        model: Model instance to validate.
        model_label: Readable label used in error messages.

    Returns:
        None. Raises ValueError if any floating parameter is not float32.
    """

    for parameter_name, parameter in model.named_parameters():
        if torch.is_floating_point(parameter.data) and parameter.data.dtype != torch.float32:
            raise ValueError(
                "Detected non-FP32 parameter despite strict FP32 requirement | "
                f"model={model_label} | parameter={parameter_name} | dtype={parameter.data.dtype}"
            )


def validate_parameter_compatibility(
    base_model: AutoModelForCausalLM,
    task_model: AutoModelForCausalLM,
    task_name: str,
) -> None:
    """Validate key/shape compatibility between base and task models.

    Args:
        base_model: Base checkpoint model.
        task_model: Task checkpoint model.
        task_name: Task key used in diagnostics.

    Returns:
        None. Raises ValueError when mismatch is detected.
    """

    base_params = dict(base_model.named_parameters())
    task_params = dict(task_model.named_parameters())

    if set(base_params.keys()) != set(task_params.keys()):
        missing_in_task = sorted(set(base_params.keys()) - set(task_params.keys()))
        missing_in_base = sorted(set(task_params.keys()) - set(base_params.keys()))
        raise ValueError(
            "Named parameter keys mismatch across base/task models. "
            f"task={task_name}, missing_in_task={missing_in_task[:5]}, missing_in_base={missing_in_base[:5]}"
        )

    for parameter_name, base_parameter in base_params.items():
        if base_parameter.shape != task_params[parameter_name].shape:
            raise ValueError(
                "Parameter shape mismatch across base/task models. "
                f"task={task_name}, parameter={parameter_name}, "
                f"base_shape={tuple(base_parameter.shape)}, "
                f"task_shape={tuple(task_params[parameter_name].shape)}"
            )


def build_task_vector_cpu(
    base_model: AutoModelForCausalLM,
    task_model: AutoModelForCausalLM,
) -> Dict[str, torch.Tensor]:
    """Build task vector on CPU with the same definition as notebook 08.

    Definition preserved from `08_jwcm_v2_soft_attribution_merge_fisher_rollout_reuse.ipynb`:
        `Delta_task = theta_task - theta_base`

    Args:
        base_model: Base model used as merge anchor.
        task_model: Task-tuned model used to compute delta from base.

    Returns:
        Dictionary mapping parameter names to CPU FP32 tensors of `Delta_task`.
    """

    task_vector: Dict[str, torch.Tensor] = {}
    base_params = dict(base_model.named_parameters())
    task_params = dict(task_model.named_parameters())

    for parameter_name in tqdm(base_params.keys(), desc="Build task vector (CPU FP32)"):
        base_tensor = base_params[parameter_name].detach().to(torch.float32).cpu()
        task_tensor = task_params[parameter_name].detach().to(torch.float32).cpu()

        # Keep signed delta because task arithmetic requires direction, not absolute value.
        task_vector[parameter_name] = task_tensor - base_tensor

    return task_vector


def apply_task_arithmetic_inplace(
    base_model: AutoModelForCausalLM,
    task_vectors: Mapping[str, Mapping[str, torch.Tensor]],
    task_lambdas: Mapping[str, float],
) -> Dict[str, Any]:
    """Apply in-place task arithmetic merge to base model in FP32.

    Merge rule:
        `theta_merge = theta_base + sum_t(lambda_t * Delta_t)`

    Args:
        base_model: Base model to overwrite with merged parameters.
        task_vectors: Mapping `task_name -> parameter_name -> Delta tensor`.
        task_lambdas: Mapping `task_name -> lambda scalar`.

    Returns:
        Summary dictionary with basic merge statistics.
    """

    vector_tasks = set(task_vectors.keys())
    lambda_tasks = set(task_lambdas.keys())
    if vector_tasks != lambda_tasks:
        raise ValueError(
            "Task mismatch between task_vectors and task_lambdas. "
            f"vector_tasks={sorted(vector_tasks)}, lambda_tasks={sorted(lambda_tasks)}"
        )

    base_params = dict(base_model.named_parameters())
    updated_parameter_count = 0
    skipped_non_floating_count = 0

    with torch.no_grad():
        for parameter_name, base_parameter in tqdm(base_params.items(), desc="Apply task arithmetic"):
            if not torch.is_floating_point(base_parameter.data):
                skipped_non_floating_count += 1
                continue

            base_fp32 = base_parameter.data.detach().to(torch.float32)
            merged_tensor = base_fp32.clone()

            # Add each task contribution independently for transparent lambda control.
            for task_name, task_vector in task_vectors.items():
                merged_tensor.add_(float(task_lambdas[task_name]) * task_vector[parameter_name].to(torch.float32))

            base_parameter.data.copy_(merged_tensor.to(torch.float32))
            updated_parameter_count += 1

    total_l1_norms: Dict[str, float] = {}
    for task_name, task_vector in task_vectors.items():
        # Aggregate L1 norm is a cheap sanity metric for how large each task update is.
        total_l1_norms[task_name] = float(
            sum(float(tensor.abs().sum().item()) for tensor in task_vector.values())
        )

    return {
        "updated_parameter_count": int(updated_parameter_count),
        "skipped_non_floating_count": int(skipped_non_floating_count),
        "task_lambdas": {k: float(v) for k, v in task_lambdas.items()},
        "task_vector_total_l1_norm": total_l1_norms,
    }


def build_output_dir_name(task_lambdas: Mapping[str, float]) -> str:
    """Build output directory name that explicitly includes task lambda values.

    Args:
        task_lambdas: Mapping from task key to lambda scalar.

    Returns:
        Directory name string with lambda tokens included.
    """

    if "if" not in task_lambdas or "math" not in task_lambdas:
        raise ValueError("Both 'if' and 'math' lambdas are required for output naming.")

    if_token = format_float_token(task_lambdas["if"])
    math_token = format_float_token(task_lambdas["math"])
    return f"task_arithmetic_if_l{if_token}_math_l{math_token}_fp32"


def save_merged_artifacts(
    model: AutoModelForCausalLM,
    tokenizer: AutoTokenizer,
    output_dir: Path,
    metadata: Mapping[str, Any],
) -> None:
    """Save merged model/tokenizer and metadata to output directory.

    Args:
        model: Merged model in FP32.
        tokenizer: Tokenizer to save alongside checkpoint.
        output_dir: Destination checkpoint directory.
        metadata: JSON-serializable metadata payload.

    Returns:
        None. Artifacts are persisted to disk.
    """

    output_dir.mkdir(parents=True, exist_ok=True)

    # Save model weights in safe tensors format while preserving FP32 parameters.
    model.save_pretrained(output_dir, safe_serialization=True)
    tokenizer.save_pretrained(output_dir)
    save_json(dict(metadata), output_dir / "merge_metadata.json")

In [ ]:
set_seed(RUNTIME.seed)

if RUNTIME.model_dtype != torch.float32:
    raise ValueError(
        "Runtime dtype changed unexpectedly. "
        f"Expected torch.float32, got {RUNTIME.model_dtype}"
    )

task_lambdas: Dict[str, float] = {
    "if": float(RUNTIME.lambda_if),
    "math": float(RUNTIME.lambda_math),
}

expected_tasks = {task_spec.name for task_spec in TASK_SPECS}
if set(task_lambdas.keys()) != expected_tasks:
    raise ValueError(
        "task_lambdas keys must exactly match TASK_SPECS names. "
        f"expected={sorted(expected_tasks)}, got={sorted(task_lambdas.keys())}"
    )

print("Loading base model in FP32...")
merge_base_model, merge_tokenizer = load_causal_lm_fp32(
    model_name_or_path=RUNTIME.base_model_id,
    device=RUNTIME.device,
)
assert_model_float32(merge_base_model, model_label="base_model")

merge_task_vectors: Dict[str, Dict[str, torch.Tensor]] = {}
merge_task_model_paths: Dict[str, str] = {}

for task_spec in TASK_SPECS:
    print(f"Loading task model in FP32 | task={task_spec.name}")
    task_model, _ = load_causal_lm_fp32(
        model_name_or_path=task_spec.model_path,
        device=RUNTIME.device,
    )
    assert_model_float32(task_model, model_label=f"task_model:{task_spec.name}")

    validate_parameter_compatibility(
        base_model=merge_base_model,
        task_model=task_model,
        task_name=task_spec.name,
    )

    merge_task_vectors[task_spec.name] = build_task_vector_cpu(
        base_model=merge_base_model,
        task_model=task_model,
    )
    merge_task_model_paths[task_spec.name] = str(task_spec.model_path)

    # Release task model once its task vector is built to reduce memory pressure.
    del task_model
    gc.collect()

merge_summary = apply_task_arithmetic_inplace(
    base_model=merge_base_model,
    task_vectors=merge_task_vectors,
    task_lambdas=task_lambdas,
)

# Re-check merged model dtype before saving to ensure FP32 requirement is preserved.
assert_model_float32(merge_base_model, model_label="merged_model")

output_dir_name = build_output_dir_name(task_lambdas=task_lambdas)
output_dir = RUNTIME.output_root / output_dir_name

metadata = {
    "created_at": now_iso(),
    "method": "task_arithmetic",
    "formula": "theta_merge = theta_base + lambda_if * Delta_if + lambda_math * Delta_math",
    "task_vector_definition": "Delta_task = theta_task - theta_base",
    "runtime": {
        "base_model_id": RUNTIME.base_model_id,
        "output_root": str(RUNTIME.output_root),
        "seed": int(RUNTIME.seed),
        "model_dtype": str(RUNTIME.model_dtype),
        "merge_device": str(RUNTIME.device),
    },
    "task_models": merge_task_model_paths,
    "task_lambdas": {k: float(v) for k, v in task_lambdas.items()},
    "merge_summary": merge_summary,
}

save_merged_artifacts(
    model=merge_base_model,
    tokenizer=merge_tokenizer,
    output_dir=output_dir,
    metadata=metadata,
)

run_summary_path = (
    RUNTIME.output_root
    / "metadata"
    / f"run_summary_{format_float_token(task_lambdas['if'])}_{format_float_token(task_lambdas['math'])}.json"
)
save_json(metadata, run_summary_path)

print(f"Saved FP32 task arithmetic checkpoint: {output_dir}")
print(f"Saved run summary: {run_summary_path}")